# Classic + Sparse Slow-Fast Demo

This notebook is a one-click slow-fast demo for the repository.

It is designed to:

1. reuse cached validation runs when available, or rerun a compact demo plan,
2. compare raw, delay, fastslow, theory_fastslow, and factor coordinates,
3. summarize predictive lift, Markov closure, spectral preservation, and Koopman-style invariance,
4. surface mined factors and cautious theory takeaways for review.

The notebook follows the naming convention `<scope>_sf_demo.ipynb` and is stored under `notebooks/sf/`.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from fsrc_sindy.research.demo import (
    build_factor_frequency_table,
    build_task_evidence_table,
    collect_demo_tables,
    load_benchmark_series,
    prepare_demo_runs,
    read_artifact_excerpt,
)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)
plt.style.use("seaborn-v0_8-whitegrid")


## Run Configuration

- `QUICK_DEMO=True` keeps the notebook lightweight.
- `REUSE_EXISTING=True` reuses canonical runs such as `runs/fastslow_validation/fastslow_theory_noisy` when they already exist.
- `RERUN=True` regenerates notebook-specific outputs under `runs/demo_notebook/sf/`.


In [ ]:
QUICK_DEMO = True
REUSE_EXISTING = True
RERUN = False
SEED = 123
GRID_MODE = "quick"
FULL_LIBRARY_SEARCH = True
OUT_ROOT = ROOT / "runs" / "demo_notebook" / "sf"

run_records = prepare_demo_runs(
    quick=QUICK_DEMO,
    reuse_existing=REUSE_EXISTING,
    rerun=RERUN,
    out_root=OUT_ROOT,
    seed=SEED,
    grid_mode=GRID_MODE,
    full_library_search=FULL_LIBRARY_SEARCH,
)

run_manifest = pd.DataFrame(run_records)
run_manifest


## Load Dashboard Tables

The helper module reads the standard research-loop artifacts and tags them by notebook demo label.

In [ ]:
dashboard = collect_demo_tables(run_records)

benchmark_summary = dashboard["benchmark_summary"].copy()
coordinate_summary = dashboard["coordinate_summary"].copy()
factor_summary = dashboard["factor_summary"].copy()
benchmark_results = dashboard["benchmark_results"].copy()
coordinate_results = dashboard["coordinate_results"].copy()
validation_gate = dashboard["validation_gate"].copy()

evidence_table = build_task_evidence_table(
    benchmark_summary=benchmark_summary,
    coordinate_summary=coordinate_summary,
    factor_summary=factor_summary,
    validation_gate=validation_gate,
)
factor_frequency = build_factor_frequency_table(factor_summary)

print(f"tasks in benchmark summary: {benchmark_summary['task'].nunique() if not benchmark_summary.empty else 0}")
print(f"tasks in coordinate summary: {coordinate_summary['task'].nunique() if not coordinate_summary.empty else 0}")
print(f"tasks in factor summary: {factor_summary['task'].nunique() if not factor_summary.empty else 0}")


## Benchmark Lift

This is the high-level predictive view. Positive gain means the fast-slow readout improved over the raw baseline at RMSE@50.

In [ ]:
benchmark_cols = [
    "demo_label",
    "task",
    "rc_fastslow_gain_pct",
    "ngrc_fastslow_gain_pct",
    "best_fastslow_variant",
    "best_factor_variant",
    "best_overall_variant",
    "best_overall_rmse50",
]
benchmark_view = benchmark_summary[benchmark_cols].sort_values(["demo_label", "task"]).reset_index(drop=True)
benchmark_view


In [ ]:
plot_cols = [
    "rc_raw_rmse50",
    "rc_fastslow_rmse50",
    "ngrc_raw_rmse50",
    "ngrc_fastslow_rmse50",
    "best_factor_rmse50",
]
pretty_cols = {
    "rc_raw_rmse50": "RC raw",
    "rc_fastslow_rmse50": "RC fast-slow",
    "ngrc_raw_rmse50": "NGRC raw",
    "ngrc_fastslow_rmse50": "NGRC fast-slow",
    "best_factor_rmse50": "best factor",
}
plot_df = benchmark_summary[["task", *plot_cols]].set_index("task").rename(columns=pretty_cols)
ax = plot_df.plot(kind="bar", figsize=(12, 4))
ax.set_ylabel("RMSE@50")
ax.set_title("Benchmark ablation: raw vs fast-slow vs factor readout")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()


## Variant-Level Ablation

This view keeps the model-level rows so expert review can inspect one-step error, rollout error, and the selected factor names.

In [ ]:
variant_rows = benchmark_results[
    benchmark_results["base_model_name"].isin(
        [
            "rc_raw",
            "rc_fastslow_readout",
            "ngrc_raw",
            "ngrc_fastslow_readout",
            "rc_factor_readout",
            "ngrc_factor_readout",
        ]
    )
][
    [
        "demo_label",
        "task",
        "variant",
        "rmse@50",
        "one_step_rmse",
        "psd_rmse",
        "readout_identifier_kind",
        "readout_factor_names",
    ]
].sort_values(["demo_label", "task", "rmse@50"]).reset_index(drop=True)
variant_rows


## Coordinate Diagnostics

These tables focus on the theory lenses: Markov closure, spectral preservation, and Koopman-style invariance.

In [ ]:
coordinate_cols = [
    "demo_label",
    "task",
    "best_closure_coordinate",
    "best_spectral_coordinate",
    "best_koopman_coordinate",
    "best_fastslow_coordinate",
    "fastslow_wins",
    "fastslow_markov_gain_ratio",
    "fastslow_spectral_corr",
    "fastslow_koopman_score",
]
coordinate_summary[coordinate_cols].sort_values(["demo_label", "task"]).reset_index(drop=True)


In [ ]:
coordinate_matrix = coordinate_results[
    [
        "demo_label",
        "task",
        "coordinate",
        "markov_gain_ratio",
        "spectral_radius_rmse",
        "spectral_radius_corr",
        "koopman_invariance_score",
    ]
].sort_values(["demo_label", "task", "coordinate"]).reset_index(drop=True)
coordinate_matrix


## Theory Evidence Table

The evidence table is intentionally conservative. It marks support only when predictive lift and coordinate-level evidence point in the same direction.

In [ ]:
evidence_view = evidence_table[
    [
        "demo_label",
        "task",
        "best_gain_pct",
        "fastslow_validation_allowed",
        "fastslow_wins",
        "claim_status",
        "confidence",
        "takeaway",
    ]
].sort_values(["demo_label", "task"]).reset_index(drop=True)
evidence_view


In [ ]:
lines = ["## Theory takeaways"]
for row in evidence_view.itertuples(index=False):
    lines.append(
        f"- `{row.task}`: {row.takeaway} Claim status: `{row.claim_status}`. Confidence: `{row.confidence}`."
    )
display(Markdown("\n".join(lines)))


## Factor Mining View

This section keeps both task-level selected factor sets and an aggregate factor frequency table.

In [ ]:
factor_cols = [
    "demo_label",
    "task",
    "identifier_kind",
    "selected_factors",
    "selected_koopman_score",
    "final_rmse50",
    "test_rmse50",
]
factor_summary[factor_cols].sort_values(["demo_label", "task"]).reset_index(drop=True)


In [ ]:
factor_frequency


## Observed Series Snapshots

The benchmark series artifacts only store the observed scalar trajectory partitions, which are enough for quick expert-side sanity checks.

In [ ]:
series_specs = [
    ("classic_noisy", "hindmarsh_rose_bursting_noisy"),
    ("sparse_observation", "lorenz96_twoscale_sparse_triplet_noisy"),
]

fig, axes = plt.subplots(len(series_specs), 1, figsize=(12, 6), sharex=False)
if len(series_specs) == 1:
    axes = [axes]

for ax, (label, task) in zip(axes, series_specs):
    record = next((item for item in run_records if item["label"] == label), None)
    if record is None:
        ax.set_visible(False)
        continue
    series = load_benchmark_series(record["run_dir"], task)
    y = series["y"]
    ax.plot(y[: min(len(y), 800)], linewidth=1.0)
    ax.set_title(f"{task}: observed scalar trajectory")
    ax.set_ylabel("observation")

axes[-1].set_xlabel("time step")
plt.tight_layout()
plt.show()


## Artifact Excerpts

The notebook also exposes the compact research-loop text artifacts so expert review can start from the generated evidence rather than from raw CSVs.

In [ ]:
for record in run_records:
    display(Markdown(f"### {record['label']}"))
    theory_excerpt = read_artifact_excerpt(record["run_dir"], "theory_evidence.md", max_lines=18)
    loop_excerpt = read_artifact_excerpt(record["run_dir"], "loop_summary.md", max_lines=18)
    display(Markdown("**Theory evidence excerpt**"))
    display(Markdown("```text\n" + theory_excerpt + "\n```"))
    display(Markdown("**Loop summary excerpt**"))
    display(Markdown("```text\n" + loop_excerpt + "\n```"))


## Suggested Next Moves

- Set `QUICK_DEMO=False` to add `vanderpol_relaxation_noisy` and `lorenz96_twoscale_sparse_mixed_noisy`.
- Set `RERUN=True` to regenerate notebook-specific outputs under `runs/demo_notebook/sf/`.
- Rename the notebook by the same `<scope>_sf_demo.ipynb` rule if you split this suite into system-specific demos later.